In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import pandas as pd
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from mordred import Calculator, descriptors as mordred_descriptors
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from assets.functions import *
from assets.benchmark import *

df = pd.read_csv("./data/caco2data.csv")
df = df.rename(columns={"target": "Caco2Papp"}).copy()

# Ensure numeric target
df["Caco2Papp"] = pd.to_numeric(df["Caco2Papp"], errors="coerce")
df = df.dropna(subset=["Caco2Papp", "smiles"])
# Apply the scaffold computation for each SMILES
df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

results_df = run_descriptor_benchmark(train_df, val_df, test_df, target_col="Caco2Papp")

results_df.sort_values("Test_MAE")

/home/tanush/anaconda3/envs/mordred2/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Training set size: 2396
Test set size: 299
Validation set size: 300

🔹 Featurizing with FCFP6...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with MACCS...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with RDKIT...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with MORDRED...


/home/tanush/anaconda3/envs/mordred2/lib/python3.9/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB


,Descriptor,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
10,MORDRED,SVM,0.427085,0.713321,0.940171,0.395295,0.621614,0.951698
6,RDKIT,RF,0.445746,0.764000,0.931368,0.397951,0.694445,0.939716
8,RDKIT,XGB,0.400672,0.668988,0.947377,0.398282,0.676718,0.942754
4,MACCS,SVM,0.500205,0.864604,0.912103,0.402867,0.644028,0.948151
7,RDKIT,SVM,0.451884,0.758525,0.932348,0.417360,0.705409,0.937797
11,MORDRED,XGB,0.434022,0.753520,0.933238,0.422419,0.657606,0.945942
9,MORDRED,RF,0.486853,0.801900,0.924390,0.460565,0.710902,0.936825
3,MACCS,RF,0.482961,0.865202,0.911981,0.468338,0.831728,0.913525
0,FCFP6,RF,0.506994,0.812597,0.922359,0.480860,0.854847,0.908651
2,FCFP6,XGB,0.530185,0.845195,0.916004,0.488093,0.738130,0.931893
